In [ ]:
from pathlib import Path
import cartopy
import cartopy.crs as ccrs
import cf_xarray  # noqa: F401
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
import rasterio
import rioxarray as rio
import xarray as xr
from rasterio.enums import Resampling
from rasterio.crs import CRS
import netCDF4
import h5netcdf
import os
import re
import pandas as pd

In [ ]:
# Define the directory paths for MODIS and PACE L2 data granules
# Note: Update these paths to the actual locations of your data

L2_dir_modis= '/Volumes/Extreme SSD/PhD backup/Satellite Data/MODIS_L2'
L2_dir_pace= '/Volumes/Extreme SSD/PhD backup/Satellite Data/PACE_L2'

pace_filenames = [f for f in os.listdir(L2_dir_pace) if f.endswith('.nc') and not f.startswith('._')]
modis_filenames = [f for f in os.listdir(L2_dir_modis) if f.endswith('.nc') and not f.startswith('._')]

#print(pace_filenames)

print(len(pace_filenames))

print(len(modis_filenames))

pace_paths = [os.path.join(L2_dir_pace, f) for f in pace_filenames ]
modis_paths = [os.path.join(L2_dir_modis, f) for f in modis_filenames ]



In [ ]:
def plot_features(ax, gridline_alpha=1):
    ax.gridlines(draw_labels={"left": "y", "bottom": "x"}, linewidth=0.25, alpha=gridline_alpha)
    ax.coastlines(linewidth=0.5)
    ax.add_feature(cartopy.feature.OCEAN, edgecolor="w", linewidth=0.01)
    ax.add_feature(cartopy.feature.LAND, edgecolor="w", linewidth=0.01)
    ax.add_feature(cartopy.feature.LAKES, edgecolor="w", linewidth=0.01)

In [ ]:
# L2 data is structured in groups, so we need to open the file as a datatree and merge the relevant groups into a single dataset
# this function basically creates a analysis ready dataset.
# L2 data has a common ground:
# /geophysical_data/Rrs     → dims: (number_of_lines, pixels_per_line, wavelength)
# /navigation_data/latitude → dims: (number_of_lines, pixels_per_line)
# /navigation_data/longitude→ dims: (number_of_lines, pixels_per_line)
# --> we want to merge these into a single dataset with dimensions (number_of_lines, pixels_per_line) and coordinates latitude, longitude, wavelength, 
# and variable Rrs(number_of_lines, pixels_per_line, wavelength). Because all three are spanned across numberoflines and pixels_per_line, we can just use merge. 
# xarray defaults to merging on common dimensions, so the lat/lon will automatically be connected to the Rrs variable as coordinates. The wavelength coordinate is also merged in, but it is not spanned across number_of_lines and pixels_per_line, so it will just be added as a coordinate without affecting the dimensions of the dataset.



def open_l2(fpath, target_wls=None):
    """ 
    Opens a PACE L2 file as an xarray dataset, assigning lat/lons (and wavelength, 
        if the dataset is 3D) as coordinates.
    Args:
        fpath - path to a L2 PACE file 
    Returns:
        ds - xarray dataset
    """
    # Open the file as a datatree to access the different groups
    # avoids using xarray's open_dataset(path, group = 'navigation_data') which doesn't handle the group structure of the file well
    # if opened with open_dataset, every group has to be handled seperately - there is no connection between groups, so lat/lons are not coordinates but just variables in the navigation_data group, and wavelength is not a coordinate but just a variable in the sensor_band_parameters group
    dt = xr.open_datatree(fpath, engine="h5netcdf", decode_timedelta=False)

    #print(f"Opened datatree with groups: {list(dt.groups)}")
    #print(f'this is the raw datatree: {dt}')
    
    try:

        #print(f'before creating geo_ds from dt["geophysical_data"]: {dt["geophysical_data"]}')

        geo_ds = dt["geophysical_data"].to_dataset()

        #print(f'geo_ds after creating from dt["geophysical_data"]: {geo_ds}')


        if target_wls is not None:
            
            #assigning the coordinate for a wavelength 
            #this takes the 172 values (dimension of wavelength) of sensor band parameters which holds the nanometer values. wls_coords then is a 172 array with wavelength values.
            wls_coord = dt["sensor_band_parameters"].ds["wavelength"]
            #dt.ds is accessing the subgroup and displaying /converting content as a dataset. this is data structure.
            #the dataset is like a ttribute to the datatree.

            #this is the structure of the datatree:
            # datatree/
            #   ├── root (dt.ds)                    ← metadata (root group)
            #   ├── geophysical_data (dt[...].ds)   ← Rrs(..., wavelength: 172)
            #   ├── sensor_band_parameters (...)    ← wavelength=[346..719]
            #   └── navigation_data (...)           ← lat, lon

            #dimension is indexes: accesible with isel(wavelength=0) while sel(wavelength=470, method='nearest') searches for coordinate value.
            
            # Use sel(method='nearest') on coord FIRST
            sel_wls = wls_coord.sel(wavelength=target_wls, method='nearest')

            print(f"Selected wavelengths: {sel_wls.values}")

            geo_ds = geo_ds.assign_coords(wavelength=wls_coord)
            geo_ds = geo_ds.sel(wavelength=sel_wls, method='nearest')  # or isel

            print(f"Geo dataset wavelengths: {geo_ds.wavelength.values}")
            
            sensor_coords = dt["sensor_band_parameters"].to_dataset().sel(wavelength=sel_wls).coords
            
        else:
            sensor_coords = dt["sensor_band_parameters"].to_dataset().coords

        ds = xr.merge((
            #dt.ds,  # this is the root group, with metadata
            dt.ds,

            #pulls actuall variables, but still without generic dimenstions: still number_of_lines, pixels_per_line, etc.
            geo_ds,

            #extracts wavelength axis etc and sets it as coordinates, but still with the original dimensions
            sensor_coords,  # safer than .coords directly

            #navigation data has lat/lon per number_of_lines and pixels_per_line, so we need to connect them as latitude and longitude coordinates to the geophysical data
            dt["navigation_data"].ds.set_coords(("longitude", "latitude")).coords,
        ))

    except:
        #this handles data which does not have the sensor_band_parameters group, so no wavelength coordinate, but is still useful for testing. this is for MODIS L2 data, which has the same structure but no sensor_band_parameters group
        ds = xr.merge((
            dt.ds,
            dt["geophysical_data"].to_dataset(),
            dt["navigation_data"].ds.set_coords(("longitude", "latitude")).coords,
        ))

    return ds

pace = open_l2(pace_paths[20], target_wls=[470, 550, 660])
pace_2 = open_l2(pace_paths[21], target_wls=[470, 550, 660])
modis = open_l2(modis_paths[1], target_wls=[470, 550, 660])

#print(pace)

target_wls = [470, 550, 660]

#pace_subset = pace.sel(wavelength=target_wls, method="nearest")

print(pace)

In [ ]:
# creating RGB ready dataset from L2 granules # THIS IS THE ORIGINAL

directory = L2_dir_pace
print(directory)

from joblib import Parallel, delayed
import re
import numpy as np
import pandas as pd
import xarray as xr

def grid_data(src, resolution, dst_crs="epsg:4326", resampling=Resampling.nearest):
    
    #print(f'this is lat_min before gridding: {src["latitude"].min().values}')
    #rint(f'this is lat_max before gridding: {src["latitude"].max().values}')
    #print(f'this is lon_min before gridding: {src["longitude"].min().values}')
    #print(f'this is lon_max before gridding: {src["longitude"].max().values}')

    if (len(list(src.dims)) == 3) and (list(src.dims)[0] != "wavelength_3d"):
        src = src.transpose("wavelength_3d", ...)

    lon_2d = np.array(src["longitude"])
    lat_2d = np.array(src["latitude"])
    n_y, n_x = lon_2d.shape

    src = src.drop_vars(["longitude", "latitude"])
    src = src.rename_dims({"pixels_per_line": "x", "number_of_lines": "y"})
    src = src.assign_coords(x=np.arange(n_x), y=np.arange(n_y))
    src = src.rio.write_crs("epsg:4326")

    # KEY: pass src_geoloc_array directly — skip left/bottom/right/top
    defaults = rasterio.warp.calculate_default_transform(
        "epsg:4326",
        dst_crs,
        n_x,
        n_y,
        src_geoloc_array=(lon_2d, lat_2d),  # ← let rasterio compute bounds itself
    )
    #print("defaults width:", defaults[1], "height:", defaults[2])  # Verify

    transform, width, height = rasterio.warp.aligned_target(*defaults, resolution)
    #print("aligned width:", width, "height:", height)  # Should be ~3200, ~1600

    lon_da = xr.DataArray(lon_2d, dims=["y", "x"])
    lat_da = xr.DataArray(lat_2d, dims=["y", "x"])

    dst = src.rio.reproject(
        dst_crs=dst_crs,
        shape=(height, width),
        transform=transform,
        src_geoloc_array=(lon_da, lat_da),
        nodata=np.nan,
        resample=resampling,
    )
    dst["x"] = dst["x"].round(9)
    dst["y"] = dst["y"].round(9)

    dst = dst.rename({"x": "longitude", "y": "latitude"})

    #check if regridding worked.
    
    #print(f'this is lat_min after gridding: {dst["latitude"].min().values}')
    #print(f'this is lat_max after gridding: {dst["latitude"].max().values}')
    #print(f'this is lon_min after gridding: {dst["longitude"].min().values}')
    #print(f'this is lon_max after gridding: {dst["longitude"].max().values}')

    return dst

def mask_ds(ds, flag="CLDICE", reverse=False):
    """
    Mask a PACE dataset for an L2 flag. Default is to mask for clouds
    Args:
        ds - xarray dataset containing "l2_flags" variable
        flag - l2 flag to mask for (see https://oceancolor.gsfc.nasa.gov/resources/atbd/ocl2flags/)
        reverse - keep only pixels with the desired flag. Default is False. E.g., use the
                  "LAND" flag to mask water pixels. 
    Returns:
        Masked dataset
    """
    if ds["l2_flags"].cf.is_flag_variable:
        if reverse==False:
            return ds.where(~(ds["l2_flags"].cf == flag))
        else:
            return ds.where((ds["l2_flags"].cf == flag))
    else:
        print("l2_flags not recognized as flag variable")

    return ds

# ── Single-granule worker (must be top-level for joblib/loky pickling) ────────
def process_single_granule(path, lat_min, lat_max, lon_min, lon_max,
                            resolution, wavelengths, flag, reverse):
    
    import cf_xarray  # ← add this
    import numpy as np
    import pandas as pd
    import xarray as xr
    import re

    try:
        ds = open_l2(path, target_wls=wavelengths)

        line_mask = (
            (ds.latitude >= lat_min) & (ds.latitude <= lat_max) &
            (ds.longitude >= lon_min) & (ds.longitude <= lon_max)
            
        ).any(dim="pixels_per_line")

        valid_lines = np.where(line_mask.values)[0]
        if len(valid_lines) == 0:
            print(f"No overlap, skipping {path}")
            return None

        line_min_idx, line_max_idx = valid_lines[0], valid_lines[-1]
        ds = ds.isel(number_of_lines=slice(line_min_idx, line_max_idx + 1))
        print(f"Sliced to {line_max_idx - line_min_idx + 1} lines — {path}")

        ds_masked  = mask_ds(ds, flag=flag, reverse=reverse)
        ds_masked  = ds_masked.transpose('wavelength', "number_of_lines", "pixels_per_line")
        ds_gridded = grid_data(ds_masked, resolution=resolution)

        ds_cropped = ds_gridded.sel(
            latitude=slice(lat_max, lat_min),
            longitude=slice(lon_min, lon_max)
        )

        match = re.search(r'\d{8}T\d{6}', path)
        timestamp = pd.to_datetime(match[0])

        # Attach timestamp as a scalar coordinate for tracking after merge
        ds_cropped = ds_cropped.assign_coords(granule_time=timestamp)

        return ds_cropped

    except Exception as e:
        print(f"Failed on {path}: {e}")
        return None

# ── Main merge function ───────────────────────────────────────────────────────
def granule_merge(directory, date_min, date_max, lat_min, lat_max, lon_min, lon_max,
                  resolution=(0.015, 0.015), wavelengths=None,
                  flag="CLDICE", reverse=False, n_jobs=1):

    # ── 1. Collect valid paths ────────────────────────────────────────────
    paths = []

    for filename in os.listdir(directory):
        if not filename.endswith('.nc') or filename.startswith('._'):
            continue
        match = re.search(r'\d{8}T\d{6}', filename)
        if not match:
            continue
        timestamp = pd.to_datetime(match[0])
        if pd.to_datetime(date_min) <= timestamp <= pd.to_datetime(date_max):
            full_path =os.path.join(directory, filename)
            paths.append(full_path)

    print(f"Found {len(paths)} granules in date range")

    if not paths:
        print("No granules found.")
        return None

    # ── 2. Process granules in parallel ───────────────────────────────────
    # loky = default joblib backend, spawns fresh processes (safe for xarray/numpy)
    # n_jobs=4 keeps memory ~10GB on a 16GB machine
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
        delayed(process_single_granule)(
            path, lat_min, lat_max, lon_min, lon_max,
            resolution, wavelengths, flag, reverse
        )
        for path in paths
    )

    gridded_granules = [r for r in results if r is not None]

    # ── 3. Merge ──────────────────────────────────────────────────────────
    if not gridded_granules:
        print("No valid granules to merge.")
        return None
    
    batch_size = 2  # Adjust based on memory constraints
    print(f"Merging {len(gridded_granules)} granules in batches of {batch_size}")

    #creates batches of 20  
    path_batches = [
        gridded_granules[i:i+batch_size]
        for i in range(0, len(gridded_granules), batch_size)
    ]

    batch_means = []

    for b_idx, batch in enumerate(path_batches):
        print(f"Batch {b_idx+1}/{len(path_batches)} — {len(batch)} granules")

        batch_stack = xr.concat(batch, dim="granule_time")
        batch_mean  = batch_stack.mean(dim="granule_time", skipna=True)
        batch_means.append(batch_mean)

        #deletes memory
        
        del batch, batch_stack
        import gc; gc.collect()

    # Final merge of batch means (~18 tiny arrays)
    final = xr.concat(batch_means, dim="batch").mean(dim="batch", skipna=True)

    return final



dataset = granule_merge(
    L2_dir_pace,
    date_min='20250602',
    date_max='20250606',
    lat_min=48.0,
    lat_max=60.0,
    lon_min=-6.0,
    lon_max=10.0,
)



dataset.to_netcdf('/Volumes/Extreme SSD/PhD backup/Satellite Data/merged_datasets/PACE_l2_20250602_20250606.nc')
    # this function will merge all granules within the specified date and spatial range into a single dataset. it will apply the same processing steps as above to each granule before merging. the output will be a single dataset with all the data from the selected granules, ready for analysis or visualization.

In [ ]:
for name, ds in [("pace_1", pace), ("pace_2", pace_2)]:
    print(f"\n{name}")
    print(f"  lon: {float(ds.longitude.min()):.2f} → {float(ds.longitude.max()):.2f}")
    print(f"  lat: {float(ds.latitude.min()):.2f} → {float(ds.latitude.max()):.2f}")
    # also check the WKT polygon baked into the file metadata
    print(f"  bounds: {ds.attrs.get('geospatial_bounds', 'N/A')}")

lon_overlap = (float(pace.longitude.min()) < float(pace_2.longitude.max()) and
               float(pace_2.longitude.min()) < float(pace.longitude.max()))
lat_overlap = (float(pace.latitude.min()) < float(pace_2.latitude.max()) and
               float(pace_2.latitude.min()) < float(pace.latitude.max()))

print(f"Lon overlap: {lon_overlap}, Lat overlap: {lat_overlap}")
print(f"Spatial overlap: {lon_overlap and lat_overlap}")



fig, ax = plt.subplots(figsize=(9, 5), subplot_kw={"projection": ccrs.PlateCarree()})
#ax.set_global()

plot_features(ax=ax)

pace.Rrs.sel({"wavelength":470}, method="nearest").plot(x="longitude", y="latitude", ax=ax, 
                                                          cmap="cool", vmin=0, vmax=0.005, )

pace_2.Rrs.sel({"wavelength":470}, method="nearest").plot(x="longitude", y="latitude", ax=ax, 
                                                         cmap="viridis", vmin=0, vmax=0.005)
plt.title("")
plt.show()

In [ ]:
# for regridding we need dimensions in order Z, Y, X. first lets check the dimensions of our two datasets: 
print(f"PACE 1 rrs dimensions: {pace_masked.Rrs.dims}")
print(f"PACE 2 rrs dimensions: {pace_2_masked.Rrs.dims}")

# number of lines: Y, pixels per line: X, wavelength: Z. (-> Y,X,Z) so we need to transpose the datasets 
# in this step the masks are dropped. this means we have to mask before!!

# Selecting only Rrs from the masked dataset, and transposing
pace_masked = pace_masked.transpose("wavelength", "number_of_lines", "pixels_per_line")
pace_2_masked = pace_2_masked.transpose("wavelength", "number_of_lines", "pixels_per_line")


print(f"PACE 1 rrs dimensions: {pace_masked.Rrs.dims} after masking and transposing")
print(f"PACE 2 rrs dimensions: {pace_2_masked.Rrs.dims} after masking and transposing")


In [ ]:


from joblib import Parallel, delayed
import re
import numpy as np
import pandas as pd
import xarray as xr

# ── Single-granule worker (must be top-level for joblib/loky pickling) ────────
def process_single_granule(path, lat_min, lat_max, lon_min, lon_max,
                            resolution, wavelengths, flag, reverse):
    
    import cf_xarray  # ← add this
    import numpy as np
    import pandas as pd
    import xarray as xr
    import re

    try:
        ds = open_l2(path, target_wls=wavelengths)

        line_mask = (
            (ds.latitude >= lat_min) & (ds.latitude <= lat_max) &
            (ds.longitude >= lon_min) & (ds.longitude <= lon_max)
        ).any(dim="pixels_per_line")

        valid_lines = np.where(line_mask.values)[0]
        if len(valid_lines) == 0:
            print(f"No overlap, skipping {path}")
            return None

        line_min_idx, line_max_idx = valid_lines[0], valid_lines[-1]
        ds = ds.isel(number_of_lines=slice(line_min_idx, line_max_idx + 1))
        print(f"Sliced to {line_max_idx - line_min_idx + 1} lines — {path}")

        ds_masked  = mask_ds(ds, flag=flag, reverse=reverse)
        ds_masked  = ds_masked.transpose("wavelength", "number_of_lines", "pixels_per_line")
        ds_gridded = grid_data(ds_masked, resolution=resolution)

        ds_cropped = ds_gridded.sel(
            latitude=slice(lat_max, lat_min),
            longitude=slice(lon_min, lon_max)
        )

        match = re.search(r'\d{8}T\d{6}', path)
        timestamp = pd.to_datetime(match[0])

        # Attach timestamp as a scalar coordinate for tracking after merge
        ds_cropped = ds_cropped.assign_coords(granule_time=timestamp)

        return ds_cropped

    except Exception as e:
        print(f"Failed on {path}: {e}")
        return None
    

# ── Main merge function ───────────────────────────────────────────────────────
def granule_merge(directory, date_min, date_max, lat_min, lat_max, lon_min, lon_max,
                  resolution=(0.015, 0.015), wavelengths=None,
                  flag="CLDICE", reverse=False, n_jobs=2):

    # ── 1. Collect valid paths ────────────────────────────────────────────
    paths = []

    for filename in directory:
        if not filename.endswith('.nc') or filename.startswith('._'):
            continue
        match = re.search(r'\d{8}T\d{6}', filename)
        if not match:
            continue
        timestamp = pd.to_datetime(match[0])
        if pd.to_datetime(date_min) <= timestamp <= pd.to_datetime(date_max):
            paths.append(filename)

    print(f"Found {len(paths)} granules in date range")

    if not paths:
        print("No granules found.")
        return None

    # ── 2. Process granules in parallel ───────────────────────────────────
    # loky = default joblib backend, spawns fresh processes (safe for xarray/numpy)
    # n_jobs=4 keeps memory ~10GB on a 16GB machine
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
        delayed(process_single_granule)(
            path, lat_min, lat_max, lon_min, lon_max,
            resolution, wavelengths, flag, reverse
        )
        for path in paths
    )

    gridded_granules = [r for r in results if r is not None]

    # ── 3. Merge ──────────────────────────────────────────────────────────
    if not gridded_granules:
        print("No valid granules to merge.")
        return None
    
    batch_size = 15  # Adjust based on memory constraints
    print(f"Merging {len(gridded_granules)} granules in batches of {batch_size}")

    #creates batches of 20  
    path_batches = [
        gridded_granules[i:i+batch_size]
        for i in range(0, len(gridded_granules), batch_size)
    ]

    batch_means = []

    for b_idx, batch in enumerate(path_batches):
        print(f"Batch {b_idx+1}/{len(path_batches)} — {len(batch)} granules")

        batch_stack = xr.concat(batch, dim="granule_time")
        batch_mean  = batch_stack.mean(dim="granule_time", skipna=True)
        batch_means.append(batch_mean)

        #deletes memory
        
        del batch, batch_stack
        import gc; gc.collect()

    # Final merge of batch means (~18 tiny arrays)
    final = xr.concat(batch_means, dim="batch").mean(dim="batch", skipna=True)

    return final


    



dataset_modis = granule_merge(
    modis_paths,
    date_min='20260717',
    date_max='20260724',
    lat_min=46,    # was -28
    lat_max=68,    # was 12
    lon_min=-25,   # was 11
    lon_max=8,    
)





#dataset_pace.to_netcdf("/Volumes/Extreme SSD/PhD backup/Satellite Data/merged_datasets/pace_merged_20260717_0724.nc")
dataset_modis.to_netcdf("/Volumes/Extreme SSD/PhD backup/Satellite Data/merged_datasets/modis_merged_20260717_0724.nc")
    # this function will merge all granules within the specified date and spatial range into a single dataset. it will apply the same processing steps as above to each granule before merging. the output will be a single dataset with all the data from the selected granules, ready for analysis or visualization.

In [ ]:

dataset_pace = granule_merge(
    pace_paths,
    date_min='20260717',
    date_max='20260724',
    lat_min=46,    # was -28
    lat_max=68,    # was 12
    lon_min=-25,   # was 11
    lon_max=8,    
)

dataset_pace.to_netcdf("/Volumes/Extreme SSD/PhD backup/Satellite Data/merged_datasets/pace_merged_20260717_0724.nc")
